In [2]:
# !pip install -q pykeops

: 

In [3]:
# Cell 1: basic environment + repo setup
import os
import sys
from pathlib import Path

!pip install -q wfdb resampy ishneholterlib pytorch-lightning

REPO_URL = "https://github.com/Anonymous0002396/CMED-Mini-Project.git"
PROJECT_ROOT = Path("/content/CMED-Mini-Project")
REPO_ROOT = PROJECT_ROOT / "SSSD-ECG"

if not PROJECT_ROOT.exists():
    !git clone {REPO_URL} /content/CMED-Mini-Project
else:
    %cd /content/CMED-Mini-Project
    !git pull

sys.path.insert(0, str(REPO_ROOT / "src"))
sys.path.insert(0, str(REPO_ROOT / "src" / "ptb_xl"))
sys.path.insert(0, str(REPO_ROOT / "src" / "sssd"))

print("PROJECT_ROOT:", PROJECT_ROOT)
print("REPO_ROOT exists:", REPO_ROOT.exists())
print("SSSD train.py exists:", (REPO_ROOT / "src" / "sssd" / "train.py").exists())
print("SSSD model exists:", (REPO_ROOT / "src" / "sssd" / "models" / "SSSD_ECG.py").exists())

/content/CMED-Mini-Project
Already up to date.
PROJECT_ROOT: /content/CMED-Mini-Project
REPO_ROOT exists: True
SSSD train.py exists: True
SSSD model exists: True


In [4]:
# Cell 2: mount Drive and extract raw PTB-XL
from google.colab import drive
from pathlib import Path
import os

drive.mount('/content/drive')

if not os.path.exists('/content/ptbxl.zip'):
    !cp "/content/drive/MyDrive/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.zip" /content/ptbxl.zip

if not os.path.exists('/content/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1'):
    !unzip -oq /content/ptbxl.zip -d /content/
    print("Extraction complete.")
else:
    print("PTB-XL already extracted, skipping extraction.")

raw_ptbxl = Path("/content/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1")
print("raw_ptbxl exists:", raw_ptbxl.exists())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PTB-XL already extracted, skipping extraction.
raw_ptbxl exists: True


In [5]:
# Cell 3: preprocess PTB-XL at 100 Hz and load processed metadata
import numpy as np
from pathlib import Path

from clinical_ts.ecg_utils import prepare_data_ptb_xl, channel_stoi_default
from clinical_ts.timeseries_utils import reformat_as_memmap, load_dataset

target_fs = 100
data_folder_ptb_xl = Path("/content/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1")
target_folder_ptb_xl = Path(f"/content/processed_ptb_xl_fs{target_fs}")

# Force rebuild for a clean notebook run
if target_folder_ptb_xl.exists():
    !rm -rf {target_folder_ptb_xl}

print("Rebuilding processed PTB-XL at:", target_folder_ptb_xl)

df_ptb_xl, lbl_itos_ptb_xl, mean_ptb_xl, std_ptb_xl = prepare_data_ptb_xl(
    data_path=data_folder_ptb_xl,
    min_cnt=0,
    target_fs=target_fs,
    channels=12,
    channel_stoi=channel_stoi_default,
    target_folder=target_folder_ptb_xl,
    recreate_data=True,
)

reformat_as_memmap(
    df_ptb_xl,
    target_folder_ptb_xl / "memmap.npy",
    data_folder=target_folder_ptb_xl,
    delete_npys=True
)

df_mapped, lbl_itos_dict, mean, std = load_dataset(target_folder_ptb_xl)

print("Processed df shape:", df_mapped.shape)
print("Available label spaces:", list(lbl_itos_dict.keys()))
print("Processed folder:", target_folder_ptb_xl)

Rebuilding processed PTB-XL at: /content/processed_ptb_xl_fs100


  0%|          | 0/21799 [00:00<?, ?it/s]

  0%|          | 0/21799 [00:00<?, ?it/s]

Processed df shape: (21799, 45)
Available label spaces: ['label_all', 'label_diag', 'label_form', 'label_rhythm', 'label_diag_subclass', 'label_diag_superclass']
Processed folder: /content/processed_ptb_xl_fs100


In [6]:
# Cell 4: create sex + MI labels on the full processed dataframe
import numpy as np
import pandas as pd

label_key = "label_all"
label_names = np.array(lbl_itos_dict[label_key])

mi_statement_names = [
    "IMI", "ASMI", "ILMI", "AMI", "ALMI",
    "INJAS", "LMI", "INJAL", "IPLMI", "IPMI",
    "INJIN", "PMI", "INJLA", "INJIL"
]

label_name_to_idx = {str(name): i for i, name in enumerate(label_names)}
mi_label_indices = [label_name_to_idx[name] for name in mi_statement_names]

def multihot_encode(indices, num_classes):
    out = np.zeros(num_classes, dtype=np.float32)
    for i in indices:
        out[i] = 1.0
    return out

def row_multihot_to_binary_mi(row_vec, mi_indices):
    return float(row_vec[mi_indices].sum() > 0)

df_real = df_mapped.copy()

# Recover full 71-dim multihot just so we can derive MI cleanly
df_real["label_multihot"] = df_real[f"{label_key}_numeric"].apply(
    lambda x: multihot_encode(x, len(label_names))
)

df_real["label_mi"] = df_real["label_multihot"].apply(
    lambda x: row_multihot_to_binary_mi(x, mi_label_indices)
).astype(np.float32)

# PTB-XL sex encoding is numeric: male=0, female=1
df_real["sex_binary"] = df_real["sex"].astype(np.float32)

print("Unique sex values:", sorted(df_real["sex_binary"].dropna().unique().tolist()))
print("Overall MI prevalence:", df_real["label_mi"].mean(), int(df_real["label_mi"].sum()), len(df_real))
print("\nCounts by sex:")
print(df_real["sex_binary"].value_counts(dropna=False).sort_index())
print("\nCounts by MI:")
print(df_real["label_mi"].value_counts(dropna=False).sort_index())
print("\nCounts by (sex, MI):")
print(df_real.groupby(["sex_binary", "label_mi"]).size())

Unique sex values: [0.0, 1.0]
Overall MI prevalence: 0.25088307 5469 21799

Counts by sex:
sex_binary
0.0    11354
1.0    10445
Name: count, dtype: int64

Counts by MI:
label_mi
0.0    16330
1.0     5469
Name: count, dtype: int64

Counts by (sex, MI):
sex_binary  label_mi
0.0         0.0         7947
            1.0         3407
1.0         0.0         8383
            1.0         2062
dtype: int64


In [7]:
# Cell 5: split train/val/test and draw a reproducible 10% stratified train subset
from sklearn.model_selection import train_test_split

max_fold_id = df_real["strat_fold"].max()

df_train_real = df_real[df_real["strat_fold"] < max_fold_id - 1].copy()
df_val_real   = df_real[df_real["strat_fold"] == max_fold_id - 1].copy()
df_test_real  = df_real[df_real["strat_fold"] == max_fold_id].copy()

print("Full split sizes:")
print("train:", len(df_train_real))
print("val:  ", len(df_val_real))
print("test: ", len(df_test_real))

# 4-way stratification key: sex x MI
# examples: "0_0", "0_1", "1_0", "1_1"
df_train_real["sex_mi_stratum"] = (
    df_train_real["sex_binary"].astype(int).astype(str)
    + "_"
    + df_train_real["label_mi"].astype(int).astype(str)
)

seed = 42
train10_idx, _ = train_test_split(
    df_train_real.index.values,
    train_size=0.10,
    random_state=seed,
    stratify=df_train_real["sex_mi_stratum"]
)

df_train_real_10 = df_train_real.loc[train10_idx].copy().sort_index()

print("\n10% train subset size:", len(df_train_real_10))

print("\nFull train counts by (sex, MI):")
print(df_train_real.groupby(["sex_binary", "label_mi"]).size())

print("\n10% subset counts by (sex, MI):")
print(df_train_real_10.groupby(["sex_binary", "label_mi"]).size())

print("\n10% subset MI prevalence:", df_train_real_10["label_mi"].mean())
print("10% subset female fraction:", (df_train_real_10["sex_binary"] == 1).mean())

Full split sizes:
train: 17418
val:   2183
test:  2198

10% train subset size: 1741

Full train counts by (sex, MI):
sex_binary  label_mi
0.0         0.0         6289
            1.0         2800
1.0         0.0         6750
            1.0         1579
dtype: int64

10% subset counts by (sex, MI):
sex_binary  label_mi
0.0         0.0         628
            1.0         280
1.0         0.0         675
            1.0         158
dtype: int64

10% subset MI prevalence: 0.25157955
10% subset female fraction: 0.4784606547960942


In [8]:
# Cell 6: align memmap dataframe and attach sex/MI conditioning
import pickle
import numpy as np

with open(target_folder_ptb_xl / "df_memmap.pkl", "rb") as f:
    df_memmap_meta = pickle.load(f)

df_memmap_meta = df_memmap_meta.copy()

# Copy aligned metadata from df_real
df_memmap_meta["sex_binary"] = df_real["sex_binary"].values.astype(np.float32)
df_memmap_meta["label_mi"] = df_real["label_mi"].values.astype(np.float32)
df_memmap_meta["strat_fold"] = df_real["strat_fold"].values

# 2-d conditioning vector: [sex_binary, label_mi]
cond_matrix = np.stack(
    [
        df_memmap_meta["sex_binary"].to_numpy(dtype=np.float32),
        df_memmap_meta["label_mi"].to_numpy(dtype=np.float32),
    ],
    axis=1,
).astype(np.float32)

df_memmap_meta["cond_sex_mi"] = list(cond_matrix)

print("df_memmap_meta shape:", df_memmap_meta.shape)
print("Example cond vectors:", df_memmap_meta["cond_sex_mi"].iloc[:5].tolist())
print("Counts by (sex, MI):")
print(df_memmap_meta.groupby(["sex_binary", "label_mi"]).size())

df_memmap_meta shape: (21799, 48)
Example cond vectors: [array([1., 0.], dtype=float32), array([0., 0.], dtype=float32), array([1., 0.], dtype=float32), array([0., 0.], dtype=float32), array([1., 0.], dtype=float32)]
Counts by (sex, MI):
sex_binary  label_mi
0.0         0.0         7947
            1.0         3407
1.0         0.0         8383
            1.0         2062
dtype: int64


In [9]:
# Cell 7: create memmap-backed split dataframes
max_fold_id = df_memmap_meta["strat_fold"].max()

# Full val/test from standard PTB-XL split
df_val_real_mem = df_memmap_meta[df_memmap_meta["strat_fold"] == max_fold_id - 1].copy()
df_test_real_mem = df_memmap_meta[df_memmap_meta["strat_fold"] == max_fold_id].copy()

# 10% train subset: use the exact same row indices selected from df_train_real
df_train_real_mem_10 = df_memmap_meta.loc[df_train_real_10.index].copy()

print("Memmap split sizes:")
print("train_10:", len(df_train_real_mem_10))
print("val:     ", len(df_val_real_mem))
print("test:    ", len(df_test_real_mem))

print("\n10% memmap subset counts by (sex, MI):")
print(df_train_real_mem_10.groupby(["sex_binary", "label_mi"]).size())

Memmap split sizes:
train_10: 1741
val:      2183
test:     2198

10% memmap subset counts by (sex, MI):
sex_binary  label_mi
0.0         0.0         628
            1.0         280
1.0         0.0         675
            1.0         158
dtype: int64


In [10]:
# Cell 8: create memmap-backed datasets with 2-d condition vectors
# import numpy as np
# np.string_ = np.bytes_  # NumPy 2 compatibility patch

from clinical_ts.timeseries_utils import TimeseriesDatasetCrops, ToTensor

input_size = 1000
tfms_ptb_xl = ToTensor()

train_real_ds_full12_10 = TimeseriesDatasetCrops(
    df_train_real_mem_10,
    output_size=input_size,
    data_folder=target_folder_ptb_xl,
    chunk_length=0,
    min_chunk_length=input_size,
    stride=input_size,
    transforms=tfms_ptb_xl,
    annotation=False,
    col_lbl="cond_sex_mi",
    memmap_filename=target_folder_ptb_xl / "memmap.npy",
)

val_real_ds_full12 = TimeseriesDatasetCrops(
    df_val_real_mem,
    output_size=input_size,
    data_folder=target_folder_ptb_xl,
    chunk_length=0,
    min_chunk_length=input_size,
    stride=input_size,
    transforms=tfms_ptb_xl,
    annotation=False,
    col_lbl="cond_sex_mi",
    memmap_filename=target_folder_ptb_xl / "memmap.npy",
)

test_real_ds_full12 = TimeseriesDatasetCrops(
    df_test_real_mem,
    output_size=input_size,
    data_folder=target_folder_ptb_xl,
    chunk_length=0,
    min_chunk_length=input_size,
    stride=input_size,
    transforms=tfms_ptb_xl,
    annotation=False,
    col_lbl="cond_sex_mi",
    memmap_filename=target_folder_ptb_xl / "memmap.npy",
)

sample_x, sample_cond = train_real_ds_full12_10[0]
print("sample ECG shape:", sample_x.shape)
print("sample cond:", sample_cond)
print("train len:", len(train_real_ds_full12_10))
print("val len:", len(val_real_ds_full12))
print("test len:", len(test_real_ds_full12))

sample ECG shape: torch.Size([12, 1000])
sample cond: tensor([1., 0.])
train len: 1741
val len: 2183
test len: 2198


In [11]:
# Cell 9: materialize the 10% train dataset into NumPy arrays

train_data_list = []
train_cond_list = []

for i in range(len(train_real_ds_full12_10)):
    x, cond = train_real_ds_full12_10[i]   # x: [12,1000], cond: [2]
    train_data_list.append(x.numpy().astype(np.float32))
    train_cond_list.append(np.asarray(cond, dtype=np.float32))

train_data_10 = np.stack(train_data_list).astype(np.float32)
train_cond_10 = np.stack(train_cond_list).astype(np.float32)

print("train_data_10 shape:", train_data_10.shape, train_data_10.dtype)
print("train_cond_10 shape:", train_cond_10.shape, train_cond_10.dtype)
print("First 5 condition vectors:")
print(train_cond_10[:5])

train_data_10 shape: (1741, 12, 1000) float32
train_cond_10 shape: (1741, 2) float32
First 5 condition vectors:
[[1. 0.]
 [1. 0.]
 [1. 0.]
 [0. 0.]
 [0. 0.]]


In [12]:
# Cell 10: save training arrays for the modified SSSD script
# from pathlib import Path
# import numpy as np

sssd_workdir = REPO_ROOT / "src" / "sssd"
sssd_workdir.mkdir(parents=True, exist_ok=True)

np.save(sssd_workdir / "ptbxl_train_data.npy", train_data_10)
np.save(sssd_workdir / "ptbxl_train_labels.npy", train_cond_10)

print("Saved:", sssd_workdir / "ptbxl_train_data.npy")
print("Saved:", sssd_workdir / "ptbxl_train_labels.npy")

# quick check
x = np.load(sssd_workdir / "ptbxl_train_data.npy", mmap_mode="r")
y = np.load(sssd_workdir / "ptbxl_train_labels.npy", mmap_mode="r")
print("Reloaded train_data:", x.shape, x.dtype)
print("Reloaded train_labels:", y.shape, y.dtype)

Saved: /content/CMED-Mini-Project/SSSD-ECG/src/sssd/ptbxl_train_data.npy
Saved: /content/CMED-Mini-Project/SSSD-ECG/src/sssd/ptbxl_train_labels.npy
Reloaded train_data: (1741, 12, 1000) float32
Reloaded train_labels: (1741, 2) float32


In [13]:
# Revised Cell 11: write a more serious sex+MI SSSD config for 10% train data
import json
from pathlib import Path

sssd_workdir = REPO_ROOT / "src" / "sssd"
config_dir = sssd_workdir / "config"
config_dir.mkdir(parents=True, exist_ok=True)

sex_mi_cfg_path = config_dir / "config_SSSD_ECG_sex_mi.json"

sex_mi_cfg = {
    "diffusion_config": {
        "T": 200,
        "beta_0": 0.0001,
        "beta_T": 0.02
    },
    "wavenet_config": {
        "in_channels": 8,
        "out_channels": 8,
        "num_res_layers": 36,
        "res_channels": 256,
        "skip_channels": 256,
        "diffusion_step_embed_dim_in": 128,
        "diffusion_step_embed_dim_mid": 512,
        "diffusion_step_embed_dim_out": 512,
        "s4_lmax": 1000,
        "s4_d_state": 64,
        "s4_dropout": 0.0,
        "s4_bidirectional": 1,
        "s4_layernorm": 1,
        "label_embed_dim": 128,
        "label_embed_classes": 2
    },
    "train_config": {
        "output_directory": "/content/sssd_sex_mi_10pct",
        "ckpt_iter": "max",
        "iters_per_ckpt": 500,
        "iters_per_logging": 50,
        "n_iters": 10000,
        "learning_rate": 2e-4,
        "batch_size": 8
    },
    "trainset_config": {
        "segment_length": 1000,
        "sampling_rate": 100,
        "finetune_dataset": "ptbxl_sex_mi_10pct"
    },
    "gen_config": {
        "output_directory": "/content/sssd_sex_mi_10pct",
        "ckpt_path": "/content/sssd_sex_mi_10pct"
    }
}

with open(sex_mi_cfg_path, "w") as f:
    json.dump(sex_mi_cfg, f, indent=2)


We do a temporary patch 

In [14]:
# Cell 12c: patch the actual hardcoded batch size line in train.py
from pathlib import Path

# Inspect the exact current trainloader line(s)
train_py = REPO_ROOT / "src" / "sssd" / "train.py"
lines = train_py.read_text().splitlines()

for i, line in enumerate(lines):
    if "trainloader" in line and "DataLoader" in line:
        print(f"LINE {i}: {repr(line)}")

LINE 87: '    trainloader = torch.utils.data.DataLoader(train_data, shuffle=True, batch_size=batch_size, drop_last=True)'


In [15]:
# Robust patch for batch_size=6 -> batch_size=batch_size
train_py = REPO_ROOT / "src" / "sssd" / "train.py"
text = train_py.read_text()

if "batch_size=batch_size" in text:
    print("Batch size already patched.")
else:
    if "batch_size=6" not in text:
        raise RuntimeError("Could not find 'batch_size=6' anywhere in train.py")
    text = text.replace("batch_size=6", "batch_size=batch_size")
    train_py.write_text(text)
    print("Patched batch size in:", train_py)

Batch size already patched.


In [18]:
!pip uninstall -y pykeops

Found existing installation: pykeops 2.3
Uninstalling pykeops-2.3:
  Successfully uninstalled pykeops-2.3


In [16]:
# import pykeops
# print(pykeops.__version__)

In [ ]:
# Cell 13: train sex+MI-conditioned SSSD on the 10% train subset
%cd /content/CMED-Mini-Project/SSSD-ECG/src/sssd
!python train.py -c config/config_SSSD_ECG_sex_mi.json

/content/CMED-Mini-Project/SSSD-ECG/src/sssd
CUDA extension for cauchy multiplication not found. Install by going to extensions/cauchy/ and running `python setup.py install`. This should speed up end-to-end training by 10-50%
Falling back on slow Cauchy kernel. Install at least one of pykeops or the CUDA extension for efficiency.
{'diffusion_config': {'T': 200, 'beta_0': 0.0001, 'beta_T': 0.02}, 'wavenet_config': {'in_channels': 8, 'out_channels': 8, 'num_res_layers': 36, 'res_channels': 256, 'skip_channels': 256, 'diffusion_step_embed_dim_in': 128, 'diffusion_step_embed_dim_mid': 512, 'diffusion_step_embed_dim_out': 512, 's4_lmax': 1000, 's4_d_state': 64, 's4_dropout': 0.0, 's4_bidirectional': 1, 's4_layernorm': 1, 'label_embed_dim': 128, 'label_embed_classes': 2}, 'train_config': {'output_directory': '/content/sssd_sex_mi_10pct', 'ckpt_iter': 'max', 'iters_per_ckpt': 500, 'iters_per_logging': 50, 'n_iters': 10000, 'learning_rate': 0.0002, 'batch_size': 8}, 'trainset_config': {'segmen

In [ ]:
# Cell B: back up checkpoints to Drive
from pathlib import Path
import os

src_dir = Path("/content/sssd_sex_mi_10pct")
dst_dir = Path("/content/drive/MyDrive/sssd_sex_mi_10pct_backup")

dst_dir.parent.mkdir(parents=True, exist_ok=True)

!mkdir -p "{dst_dir}"
!cp -r "{src_dir}"/* "{dst_dir}/"

print("Backed up checkpoints to:", dst_dir)

In [ ]:
print("test")

: 